# TalentLens บน Google Colab (ฟรี)

ทุกขั้นใน notebook นี้ **ไม่มีค่าใช้จ่าย**: ใช้ Tesseract OCR อ่านภาพสแกน และโมเดล open-source
ที่รันผ่าน [Ollama](https://ollama.com) บน GPU ฟรีของ Colab แทน Claude API

**ก่อนเริ่ม:** เมนู *Runtime → Change runtime type → T4 GPU* แล้วรันทีละช่องจากบนลงล่าง

ผลที่ได้เทียบ 3 แนวทาง + baseline

| วิธี | อ่านภาพสแกน | เข้าใจคำพ้องความหมาย |
| --- | --- | --- |
| `rules` | ไม่ได้ | ไม่ได้ |
| `ocr+rules` | ได้ (Tesseract) | ไม่ได้ |
| `ocr+<โมเดล>` | ได้ (Tesseract) | ได้ (LLM open-source) |
| `tfidf` (baseline จัดอันดับ) | ไม่ได้ | ไม่ได้ |

## 1. ดาวน์โหลดโค้ด ติดตั้ง Python packages และ Tesseract (ไทย + อังกฤษ)

In [ ]:
!git clone -q https://github.com/thassanunsat-art/Pearson-TalentLens.git
%cd Pearson-TalentLens
!pip install -q -r requirements.txt
!apt-get -qq update > /dev/null && apt-get -qq install -y tesseract-ocr tesseract-ocr-tha zstd lshw > /dev/null
!tesseract --list-langs

## 2. ติดตั้งและเปิด Ollama แล้วดาวน์โหลดโมเดล

ค่าเริ่มต้นคือ `qwen2.5:7b` (ประมาณ 4.7 GB รองรับภาษาไทย) เปลี่ยนได้ที่ตัวแปร `LOCAL_MODEL`
เช่น `qwen2.5:14b` แม่นกว่าแต่ช้ากว่า หรือ `llama3.1:8b` ดูรายชื่อได้ที่ ollama.com/library

In [ ]:
LOCAL_MODEL = "qwen2.5:7b"

!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import os, subprocess, time
os.environ["TALENTLENS_LOCAL_MODEL"] = LOCAL_MODEL
ollama = subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
!ollama pull {LOCAL_MODEL}
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

## 3. รัน tests

In [ ]:
!python -m pytest -q -rs

## 4. ลองรันเล็ก ๆ ก่อน (2 CV ต่อตำแหน่ง)

ดูบรรทัด **วินาทีต่อ CV** เพื่อประมาณเวลาของชุดเต็ม (40 CV)

In [ ]:
!python eval/run_eval.py --mode free --local-model {LOCAL_MODEL} --limit 2 --no-counterfactual

## 5. รันชุดเต็ม 40 CV

ผลของโมเดลถูก cache ไว้ใน `eval/cache/` ถ้า Colab ตัดการเชื่อมต่อกลางทาง ให้รันขั้นที่ 1–2 ใหม่
แล้วรันช่องนี้ต่อ ระบบจะทำต่อจาก CV ที่ยังไม่เสร็จ (ถ้ายังอยู่ใน session เดิม)

In [ ]:
!python eval/run_eval.py --mode free --local-model {LOCAL_MODEL} --no-counterfactual
from IPython.display import Markdown, display
display(Markdown(open("eval/results/REPORT.md", encoding="utf-8").read()))

## 6. ทดสอบความเป็นธรรม (สลับชื่อ/เพศ)

รัน CV อีก 10 คู่ ทั้งแบบปิดชื่อและไม่ปิดชื่อ (เรียกโมเดลเพิ่มราว 30 ครั้ง)

In [ ]:
!python eval/run_eval.py --mode free --local-model {LOCAL_MODEL}
display(Markdown(open("eval/results/REPORT.md", encoding="utf-8").read()))

## 7. ลองคัดกรอง CV ภาพสแกนทีละฉบับ

In [ ]:
from datetime import date
from talentlens import local_llm
from talentlens.ingest import ingest_path
from talentlens.pipeline import screen_document
from talentlens.schemas import JobSpec

local_llm.MODEL = LOCAL_MODEL
job = JobSpec.model_validate_json(open("data/jd/data_analyst.json", encoding="utf-8").read())
r = screen_document(ingest_path("data/cv/data_analyst/DA15.jpg"), job, mode="local", today=date(2026, 11, 30))
print(f"คะแนนรวม {r.total_score} · ไม่ผ่านเกณฑ์บังคับ: {r.must_have_failed or '-'} · {r.seconds} วินาที")
for s in r.criterion_scores:
    print(f"[{s.score:>3}] {s.criterion}\n      หลักฐาน: {s.evidence}  ตรวจแล้ว: {s.evidence_verified}")
print("\n--- ข้อความที่ OCR อ่านได้ (ปิดชื่อแล้ว) ---\n", r.redacted_text[:800])

## 8. ดาวน์โหลดผลประเมินกลับมา commit ใน repo

In [ ]:
!zip -qr eval_results.zip eval/results
from google.colab import files
files.download("eval_results.zip")

---
## (ไม่บังคับ) เทียบกับ Claude API - มีค่าใช้จ่าย

ข้ามส่วนนี้ได้ ถ้าต้องการเทียบ ให้ใส่ `ANTHROPIC_API_KEY` ใน Colab Secrets (ไอคอน 🔑) แล้วรันช่องด้านล่าง
ใช้ `claude-haiku-5-5` ซึ่งถูกที่สุด และลอง 2 CV ก่อนเพื่อดู "USD ต่อ CV"

In [ ]:
import os
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
!python eval/run_eval.py --mode llm --model claude-haiku-5-5 --limit 2 --no-counterfactual